## 1 · Imports

In [6]:
## 1 · Imports

import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

print("Ready")

Ready


## 2 · Load Shop Data

In [7]:
## 2 · Load Shop Data

df = pd.read_csv("csv files/raw_umbrella_data.csv")
print(f"✓ {len(df)} rows loaded")

✓ 2914 rows loaded


## 3 · Load PLUTO

In [8]:
## 3 · Load PLUTO

pluto = pd.read_csv("NYC_pluto_25v4_csv/pluto_25v4.csv", low_memory=False)
pluto_mn = pluto[pluto["borough"] == "MN"].dropna(subset=["latitude", "longitude"]).copy()
print(f"✓ {len(pluto_mn)} Manhattan lots loaded")

✓ 42116 Manhattan lots loaded


## 4. match each shop to its nearest lot directly.

In [9]:
## 5 · Plot Area — Match Each Shop to Nearest PLUTO Lot

from sklearn.neighbors import BallTree

# Filter to Manhattan only and drop rows without coordinates
pluto_mn = pluto[pluto["borough"] == "MN"].dropna(subset=["latitude", "longitude"]).copy()

# Build spatial index on PLUTO lots
pluto_coords = np.radians(pluto_mn[["latitude", "longitude"]].values)
tree = BallTree(pluto_coords, metric="haversine")

# Query nearest lot for each shop
shop_coords = np.radians(df[["lat", "lon"]].values)
distances, indices = tree.query(shop_coords, k=1)

# Attach lotarea from nearest lot
df["lot_area_sqft"] = pluto_mn.iloc[indices.flatten()]["lotarea"].values

print(f"✓ Done")
print(f"  Filled : {df['lot_area_sqft'].notna().sum()}")
print(f"  Null   : {df['lot_area_sqft'].isna().sum()}")
print(f"\n  Mean lot area : {df['lot_area_sqft'].mean():.0f} sqft")
print(f"  Min  lot area : {df['lot_area_sqft'].min():.0f} sqft")
print(f"  Max  lot area : {df['lot_area_sqft'].max():.0f} sqft")

✓ Done
  Filled : 2914
  Null   : 0

  Mean lot area : 21264 sqft
  Min  lot area : 40 sqft
  Max  lot area : 364000 sqft


## 5. Commercial & residential Ratio

In [10]:
## 5 · Commercial & Residential Ratio

matched = pluto_mn.iloc[indices.flatten()].copy()

df["com_ratio"]  = matched["comarea"].values  / matched["bldgarea"].replace(0, np.nan).values
df["res_ratio"]  = matched["resarea"].values  / matched["bldgarea"].replace(0, np.nan).values

print(f"✓ Done")
print(f"  com_ratio filled : {df['com_ratio'].notna().sum()}")
print(f"  res_ratio filled : {df['res_ratio'].notna().sum()}")

✓ Done
  com_ratio filled : 2693
  res_ratio filled : 2693


## 6. Save CSV

In [11]:
## 6 · Save Progress

df.to_csv("csv files/TS-SD-plot_area.csv", index=False, encoding="utf-8")
print(f"✓ {len(df)} records saved to 'csv files/TS-SD-plot_area.csv'")

✓ 2914 records saved to 'csv files/TS-SD-plot_area.csv'


In [ ]:
## 6 · Save Progress

df.to_csv("csv files/TS-SD-plot_area.csv", index=False, encoding="utf-8")
print(f"✓ {len(df)} records saved to 'csv files/TS-SD-plot_area.csv'")